In [1]:
from pennylane import numpy as np
import pennylane as qml
import itertools
from itertools import combinations
from scipy.optimize import minimize
from math import comb
import qaoa_aux 
#importlib.reload(qaoa_aux)

In [2]:
def U_B(beta,edge_list):
    for edge_idx, (u, v) in enumerate(edge_list):
        theta = beta[edge_idx]
        qml.S(wires=u)
        qml.SingleExcitationMinus(-2 * theta,wires=[u, v])
        qml.adjoint(qml.S)(wires=u)

In [3]:
def U_B_standard(beta, edge_list):
    for u, v in edge_list:
        qml.S(wires=u)
        qml.SingleExcitationMinus(-2 * beta, wires=[u, v])
        qml.adjoint(qml.S)(wires=u)

In [4]:
def U_C_MC(gamma,edge_list):
    edge_idx = 0
    for u, v in edge_list: 
        
        qml.CNOT(wires=(u, v))
        qml.RZ(gamma[edge_idx], wires=v)
        qml.CNOT(wires=(u, v))
        edge_idx += 1  

In [5]:
def U_C(gamma,cost_terms):
    # we have term index + cost terms so all the even numbered ZZ's
    for term_idx, (_, wires, coeff) in enumerate(cost_terms):
        theta = 2 * gamma[term_idx] * coeff
        # example for 4 consectuve z's
        #qml.PauliRot(theta, "ZZZZ", wires=[0,1,2,3])
        qml.PauliRot(theta,"Z" * len(wires),wires=list(wires))

In [6]:
def objective(gammas,betas,circuit,hyperedge_list,cost_terms,weights):
    expectation_values = circuit(gammas, betas)
    value = 0.0

    # weighted constant term for each hyperedge
    for edge_idx, edge in enumerate(hyperedge_list):
        k = len(edge)
        #w_e * 1 - (1/2^k-1)
        value += weights[edge_idx] * (1 - 1 / (2 ** (k - 1)))

    # coeff is weighted in build_cost_terms()
    for term_idx, (_, _, coeff) in enumerate(cost_terms):
        value += coeff * expectation_values[term_idx]

    return value

In [7]:
# Example small test hypergraph
# hyperedge_list = [[0, 1, 2, 3]]

# n_wires = 4
# num_layers = 1
# seed = 42
# example_weights = [1]

# rng = np.random.default_rng(seed)

# edge_list = qaoa_aux.clique_expansion(hyperedge_list)
# cost_terms =  qaoa_aux.build_cost_terms(hyperedge_list,example_weights)

# num_mixer_edges = len(edge_list)
# num_cost_terms = len(cost_terms)

# target_weight = n_wires // 2
# initial_state = qaoa_aux.make_dicke_state(n_wires, target_weight)

# dev = qml.device("default.qubit", wires=n_wires)

# print("example of cost term quadruple Z:")
# print(cost_terms[-1])

# print("Mixer edges:", edge_list)
# print("Number of mixer parameters/layer:", num_mixer_edges)
# print("Number of cost parameters/layer:", num_cost_terms)

In [8]:
def run_qaoa(
    hyperedge_list,
    n,
    weights,
    runtype,
    num_layers=1,
    seed=42,
    scaling_constant=0.1,
    step_size=0.01,
    max_trials=100
):
    n_wires = n
    assert len(weights) == len(hyperedge_list)

    # Build mixer edges and cost terms
    cost_edge_list = qaoa_aux.clique_expansion(hyperedge_list)
    mixer_edge_list = qaoa_aux.connected_clique_expansion(hyperedge_list, n)
    cost_terms = qaoa_aux.build_cost_terms(hyperedge_list,weights)

    # Balanced initial state
    target_weight = (n_wires + 1) // 2
    initial_state = qaoa_aux.make_dicke_state(n_wires, target_weight)

    dev = qml.device("lightning.qubit", wires=n_wires)
    
    @qml.qnode(dev, diff_method="adjoint",device_vjp=True)
    def aon_circuit(gammas, betas):
        qml.StatePrep(initial_state,wires=range(n_wires))

        for layer in range(num_layers):
            U_C(gammas[:, layer],cost_terms)
            U_B(betas[:, layer],mixer_edge_list)

        observables = []

        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire)for wire in wires])
            observables.append(qml.expval(z_term))
        return observables

    @qml.qnode(dev, diff_method="adjoint",device_vjp=True)
    def mc_circuit(gammas,betas):
        qml.StatePrep(initial_state,wires=range(n_wires))
        for layer in range(num_layers):
            U_C_MC(gammas[:,layer],cost_edge_list)
            U_B(betas[:, layer],mixer_edge_list)
        observables = []

        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire)for wire in wires])
            observables.append(qml.expval(z_term))
        return observables

    @qml.qnode(dev, diff_method="adjoint",device_vjp=True)
    def standard_circuit_aon(gammas, betas):
        qml.StatePrep(initial_state, wires=range(n_wires))    
        for layer in range(num_layers):
            # Multi-angle AoN cost
            U_C(gammas[:, layer], cost_terms)
            # Single-angle mixer
            U_B_standard(betas[layer], mixer_edge_list)
    
        observables = []
    
        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire) for wire in wires])
            observables.append(qml.expval(z_term))
    
        return observables

    @qml.qnode(dev, diff_method="adjoint",device_vjp=True)
    def standard_circuit_mc(gammas, betas):
        qml.StatePrep(initial_state, wires=range(n_wires))    
        for layer in range(num_layers):
            # single-angle AoN cost
            U_C_MC(gammas[:,layer],cost_edge_list)
            # Single-angle mixer
            U_B_standard(betas[layer], mixer_edge_list)
    
        observables = []
    
        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire) for wire in wires])
            observables.append(qml.expval(z_term))
    
        return observables

    

    if runtype == "aon":
        circuit = aon_circuit
        n_gamma = len(cost_terms)
        single_beta = False
    
    elif runtype == "mc":
        circuit = mc_circuit
        n_gamma = len(cost_edge_list)
        single_beta = False
    
    elif runtype == "standard_aon":
        circuit = standard_circuit_aon
        n_gamma = len(cost_terms)
        single_beta = True
    elif runtype == "standard_mc":
        circuit = standard_circuit_mc
        n_gamma = len(cost_edge_list)
        single_beta = True
    else:
        raise ValueError("runtype must be 'aon', 'mc','standard_aon', 'standard_mc'")

    # Random parameter initialization
    rng = np.random.default_rng(seed)
    
    gamma_params = np.array(scaling_constant * rng.random((n_gamma, num_layers)),requires_grad=True)
    # Single-angle or multi-angle mixer parameters
    if single_beta:
        beta_shape = (num_layers,)
    else:
        beta_shape = (len(mixer_edge_list), num_layers)
    
    beta_params = np.array(scaling_constant * rng.random(beta_shape),requires_grad=True)
    
    opt = qml.AdamOptimizer(stepsize=step_size)
    objective_values = []
    print("Gamma shape:", gamma_params.shape)
    print("Beta shape:", beta_params.shape)
    print("Total parameters:", gamma_params.size + beta_params.size)
    
    # Define optimization objective
    def loss(g, b):
        return objective(
            g,
            b,
            circuit,
            hyperedge_list,
            cost_terms,
            weights
        )
    
    initial_value = loss(gamma_params, beta_params)
    objective_values.append(float(initial_value))
    
    print(f"Initial objective = {initial_value:.6f} | Runtype = {runtype}")
    
    # Optimization
    for trial in range(max_trials):
    
        gamma_params, beta_params = opt.step(
            loss,
            gamma_params,
            beta_params
        )
    
        current_value = loss(gamma_params, beta_params)
        objective_values.append(float(current_value))
    
        if trial % 50 == 0:
            print(
                f"Iteration {trial:3d} | "
                f"objective = {current_value:.6f}"
            )
    
    return {
        "gammas": gamma_params,
        "betas": beta_params,
        "objective_values": objective_values,
        "circuit": circuit,
        "cost_edge_list": cost_edge_list,
        "mixer_edge_list": mixer_edge_list,
        "cost_terms": cost_terms,
        "n_wires": n_wires
    }

In [18]:
# P = np.load("../data/poisson_16_12_3/P/03.npy")
# w= np.load("../data/poisson_16_12_3/w/03.npy")

# hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)

In [19]:

# result = run_qaoa(
#     hyperedge_list,
#     weights=w,
#     n=12,
#     num_layers=1,
#     seed=42,
#     scaling_constant=0.1,
#     step_size=0.05,
#     max_trials=300
# )

In [20]:

# result = run_qaoa(
#     hyperedge_list,
#     weights=w,
#     runtype="mc",
#     n=12,
#     num_layers=1,
#     seed=42,
#     scaling_constant=0.1,
#     step_size=0.05,
#     max_trials=300
# )

In [21]:
import os
import glob
import pandas as pd
folder_P = "../data/poisson_16_12_3/P"
folder_w = "../data/poisson_16_12_3/w"

p_files = sorted(glob.glob(f"{folder_P}/*.npy"))[:20]

results_aon = []
results_mc=[]

for p_file in p_files:

    instance = os.path.splitext(os.path.basename(p_file))[0]
    P = np.load(p_file)
    w_file = os.path.join(folder_w,f"{instance}.npy")
    w = np.load(w_file)
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [3]:

        print(f"\nRunning instance {instance} | p={p} | Runtype=AoN")

        result_aon = run_qaoa(hyperedge_list,weights=w,runtype="aon",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        final_objective = float(result_aon["objective_values"][-1])

        initial_objective = float(result_aon["objective_values"][0])

        results_aon.append({
            "dataset": "poisson_16_12_3",
            "instance": instance,
            "runtype": "AoN",
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "gammas": np.array2string(np.asarray(result_aon["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result_aon["betas"]),separator=","),
            "initial_objective": initial_objective,
            "final_objective": final_objective 
        })
        print(f"\nRunning instance {instance} | p={p} | Runtype=AoN")

        result_mc = run_qaoa(hyperedge_list,weights=w,runtype="mc",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        final_objective = float(result_mc["objective_values"][-1])

        initial_objective = float(result_mc["objective_values"][0])

        results_mc.append({
            "dataset": "poisson_16_12_3",
            "instance": instance,
            "runtype": "MC",
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "gammas": np.array2string(np.asarray(result_mc["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result_mc["betas"]),separator=","),
            "initial_objective": initial_objective,
            "final_objective": final_objective 
        })


        
df_aon = pd.DataFrame(results_aon)
df_mc = pd.DataFrame(results_mc)
csv_path_aon = "poisson_16_12_3_min_bisection_aon.csv"
csv_path_mc = "poisson_16_12_3_min_bisection_mc.csv"

df_aon.to_csv(csv_path_aon,index=False)
df_mc.to_csv(csv_path_mc,index=False)

print(f"saved results to: {csv_path_aon} | {csv_path_mc}")


Running instance 01 | p=3 | Runtype=AoN
Initial objective = 12.899153 | Runtype = aon
Iteration   0 | objective = 12.719361
Iteration  50 | objective = 6.643126
Iteration 100 | objective = 6.119534
Iteration 150 | objective = 6.062016
Iteration 200 | objective = 6.045512
Iteration 250 | objective = 6.038051

Running instance 01 | p=3 | Runtype=AoN
Initial objective = 13.241045 | Runtype = mc
Iteration   0 | objective = 13.101572
Iteration  50 | objective = 7.370191
Iteration 100 | objective = 6.371628
Iteration 150 | objective = 6.197788
Iteration 200 | objective = 6.127094
Iteration 250 | objective = 6.086885

Running instance 02 | p=3 | Runtype=AoN
Initial objective = 11.477548 | Runtype = aon
Iteration   0 | objective = 11.284130
Iteration  50 | objective = 7.603582
Iteration 100 | objective = 6.541670
Iteration 150 | objective = 6.266166
Iteration 200 | objective = 6.164357
Iteration 250 | objective = 6.114580

Running instance 02 | p=3 | Runtype=AoN
Initial objective = 11.876925 

In [11]:
df = pd.DataFrame(results)
csv_path = "poisson_16_12_3_qaoa_p1_p2_p3.csv"

df.to_csv(csv_path,index=False)

print(f"saved results to: {csv_path}")

saved results to: poisson_16_12_3_qaoa_p1_p2_p3.csv


In [22]:
import os
import glob
import pandas as pd
folder_P = "../data/poisson_16_12_4/P"
folder_w = "../data/poisson_16_12_4/w"

p_files = sorted(glob.glob(f"{folder_P}/*.npy"))[:20]

results_aon = []
results_mc=[]

for p_file in p_files:

    instance = os.path.splitext(os.path.basename(p_file))[0]
    P = np.load(p_file)
    w_file = os.path.join(folder_w,f"{instance}.npy")
    w = np.load(w_file)
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [3]:

        print(f"\nRunning instance {instance} | p={p} | Runtype=AoN")

        result_aon = run_qaoa(hyperedge_list,weights=w,runtype="aon",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        final_objective = float(result_aon["objective_values"][-1])

        initial_objective = float(result_aon["objective_values"][0])

        results_aon.append({
            "dataset": "poisson_16_12_3",
            "instance": instance,
            "runtype": "AoN",
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "gammas": np.array2string(np.asarray(result_aon["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result_aon["betas"]),separator=","),
            "initial_objective": initial_objective,
            "final_objective": final_objective 
        })
        print(f"\nRunning instance {instance} | p={p} | Runtype=MC")

        result_mc = run_qaoa(hyperedge_list,weights=w,runtype="mc",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        final_objective = float(result_mc["objective_values"][-1])

        initial_objective = float(result_mc["objective_values"][0])

        results_mc.append({
            "dataset": "poisson_16_12_3",
            "instance": instance,
            "runtype": "MC",
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "gammas": np.array2string(np.asarray(result_mc["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result_mc["betas"]),separator=","),
            "initial_objective": initial_objective,
            "final_objective": final_objective 
        })


        
df_aon = pd.DataFrame(results_aon)
df_mc = pd.DataFrame(results_mc)
csv_path_aon = "poisson_16_12_4_min_bisection_aon.csv"
csv_path_mc = "poisson_16_12_4_min_bisection_mc.csv"

df_aon.to_csv(csv_path_aon,index=False)
df_mc.to_csv(csv_path_mc,index=False)

print(f"saved results to: {csv_path_aon} | {csv_path_mc}")


Running instance 01 | p=3 | Runtype=AoN
Initial objective = 14.105714 | Runtype = aon
Iteration   0 | objective = 13.984029
Iteration  50 | objective = 10.549595
Iteration 100 | objective = 10.127286
Iteration 150 | objective = 10.065514
Iteration 200 | objective = 10.041536
Iteration 250 | objective = 10.031259

Running instance 01 | p=3 | Runtype=MC
Initial objective = 14.367430 | Runtype = mc
Iteration   0 | objective = 14.236992
Iteration  50 | objective = 10.308067
Iteration 100 | objective = 10.123850
Iteration 150 | objective = 10.087219
Iteration 200 | objective = 10.064701
Iteration 250 | objective = 10.051385

Running instance 02 | p=3 | Runtype=AoN
Initial objective = 13.631507 | Runtype = aon
Iteration   0 | objective = 13.521116
Iteration  50 | objective = 11.118460
Iteration 100 | objective = 10.221159
Iteration 150 | objective = 10.053392
Iteration 200 | objective = 10.029494
Iteration 250 | objective = 10.019954

Running instance 02 | p=3 | Runtype=MC
Initial objective

In [14]:
import os
import glob
import pandas as pd
folder_P = "../data/poisson_16_12_5/P"
folder_w = "../data/poisson_16_12_5/w"

p_files = sorted(glob.glob(f"{folder_P}/*.npy"))[:20]

results_aon = []
results_mc=[]

for p_file in p_files:

    instance = os.path.splitext(os.path.basename(p_file))[0]
    P = np.load(p_file)
    w_file = os.path.join(folder_w,f"{instance}.npy")
    w = np.load(w_file)
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [3]:

        print(f"\nRunning instance {instance} | p={p} | Runtype=AoN")

        result_aon = run_qaoa(hyperedge_list,weights=w,runtype="aon",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        final_objective = float(result_aon["objective_values"][-1])

        initial_objective = float(result_aon["objective_values"][0])

        results_aon.append({
            "dataset": "poisson_16_12_5",
            "instance": instance,
            "runtype": "AoN",
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "gammas": np.array2string(np.asarray(result_aon["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result_aon["betas"]),separator=","),
            "initial_objective": initial_objective,
            "final_objective": final_objective 
        })
        # print(f"\nRunning instance {instance} | p={p} | Runtype=MC")

        # result_mc = run_qaoa(hyperedge_list,weights=w,runtype="mc",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        # final_objective = float(result_mc["objective_values"][-1])

        # initial_objective = float(result_mc["objective_values"][0])

        # results_mc.append({
        #     "dataset": "poisson_16_12_3",
        #     "instance": instance,
        #     "runtype": "MC",
        #     "p": p,
        #     "n_vertices": n,
        #     "n_hyperedges": len(hyperedge_list),
        #     "gammas": np.array2string(np.asarray(result_mc["gammas"]),separator=","),
        #     "betas": np.array2string(np.asarray(result_mc["betas"]),separator=","),
        #     "initial_objective": initial_objective,
        #     "final_objective": final_objective 
        # })


        
df_aon = pd.DataFrame(results_aon)
# df_mc = pd.DataFrame(results_mc)
csv_path_aon = "poisson_16_12_5_min_bisection_aon.csv"
# csv_path_mc = "poisson_16_12_4_min_bisection_mc.csv"

df_aon.to_csv(csv_path_aon,index=False)
# df_mc.to_csv(csv_path_mc,index=False)

print(f"saved results to: {csv_path_aon}")


Running instance 01 | p=3 | Runtype=AoN
Initial objective = 14.730946 | Runtype = aon
Iteration   0 | objective = 14.639105
Iteration  50 | objective = 12.183638
Iteration 100 | objective = 12.030689
Iteration 150 | objective = 12.015090
Iteration 200 | objective = 12.008995
Iteration 250 | objective = 12.005965

Running instance 02 | p=3 | Runtype=AoN
Initial objective = 15.543316 | Runtype = aon
Iteration   0 | objective = 15.503036
Iteration  50 | objective = 13.342202
Iteration 100 | objective = 13.036327
Iteration 150 | objective = 13.010706
Iteration 200 | objective = 13.004516
Iteration 250 | objective = 13.002183

Running instance 03 | p=3 | Runtype=AoN
Initial objective = 14.913949 | Runtype = aon
Iteration   0 | objective = 14.843676
Iteration  50 | objective = 13.117103


IOStream.flush timed out


Iteration 100 | objective = 13.021105
Iteration 150 | objective = 13.008622
Iteration 200 | objective = 13.004291
Iteration 250 | objective = 13.002482

Running instance 04 | p=3 | Runtype=AoN
Initial objective = 15.294995 | Runtype = aon
Iteration   0 | objective = 15.240881
Iteration  50 | objective = 13.493446
Iteration 100 | objective = 13.067014
Iteration 150 | objective = 13.025409
Iteration 200 | objective = 13.014973
Iteration 250 | objective = 13.010160

Running instance 05 | p=3 | Runtype=AoN
Initial objective = 14.335002 | Runtype = aon
Iteration   0 | objective = 14.252976
Iteration  50 | objective = 12.138893
Iteration 100 | objective = 12.013681
Iteration 150 | objective = 12.005972
Iteration 200 | objective = 12.003007
Iteration 250 | objective = 12.002606

Running instance 06 | p=3 | Runtype=AoN
Initial objective = 14.788234 | Runtype = aon
Iteration   0 | objective = 14.693868
Iteration  50 | objective = 12.383027
Iteration 100 | objective = 12.046388
Iteration 150 | o

In [9]:
import os
import glob
import pandas as pd
folder_P = "/Users/stevehalley/Code/Github/QuantumHypergraphPartitioning/data/email-Enron/numpy_15"
def get_edge_list(P):
    edges=[]
    for idx in range(P.shape[1]):
        #indices that are not 0
        edge = tuple(np.flatnonzero(P[:,idx]))
        edges.append(edge)
    return edges

p_files = sorted(glob.glob(f"{folder_P}/*.npy"))


results_aon = []
results_mc=[]

for p_file in p_files:

    instance = os.path.splitext(os.path.basename(p_file))[0]
    # Load hypergraph
    P = np.load(p_file)
    edges = get_edge_list(P)
    w = np.ones(len(edges))
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [3]:

        print(f"\nRunning instance {instance} | p={p} | Runtype=Standard Single Angle")

        result_aon = run_qaoa(hyperedge_list,weights=w,runtype="standard_mc",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        final_objective = float(result_aon["objective_values"][-1])

        initial_objective = float(result_aon["objective_values"][0])

        results_aon.append({
            "dataset": "enron",
            "instance": instance,
            "runtype": "AoN",
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "gammas": np.array2string(np.asarray(result_aon["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result_aon["betas"]),separator=","),
            "initial_objective": initial_objective,
            "final_objective": final_objective 
        })
        
df_aon = pd.DataFrame(results_aon)
# df_mc = pd.DataFrame(results_mc)
csv_path_aon = "enron_min_bisection_standardsingleangle.csv"
# csv_path_mc = "poisson_16_12_4_min_bisection_mc.csv"

df_aon.to_csv(csv_path_aon,index=False)
# df_mc.to_csv(csv_path_mc,index=False)

print(f"saved results to: {csv_path_aon}")


Running instance email-Enron_1 | p=3 | Runtype=Standard Single Angle
Gamma shape: (18, 3)
Beta shape: (3,)
Total parameters: 57
Initial objective = 8.878346 | Runtype = standard_mc
Iteration   0 | objective = 8.789108
Iteration  50 | objective = 6.237112
Iteration 100 | objective = 5.849787
Iteration 150 | objective = 5.805499
Iteration 200 | objective = 5.753024
Iteration 250 | objective = 5.678227

Running instance email-Enron_2 | p=3 | Runtype=Standard Single Angle
Gamma shape: (86, 3)
Beta shape: (3,)
Total parameters: 261
Initial objective = 43.001296 | Runtype = standard_mc
Iteration   0 | objective = 42.742485
Iteration  50 | objective = 32.129617
Iteration 100 | objective = 30.474324
Iteration 150 | objective = 30.103489
Iteration 200 | objective = 30.011166
Iteration 250 | objective = 29.989352

Running instance email-Enron_3 | p=3 | Runtype=Standard Single Angle
Gamma shape: (103, 3)
Beta shape: (3,)
Total parameters: 312
Initial objective = 133.241084 | Runtype = standard_m